# Preliminary Analysis

## Import Libraries and Shared Libraries

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.loader import load_tables
from utils.schema_guard import SchemaContract

REPORT_FILE = "../phase 1 - design/0-Preliminary Analysis.md"
md_content = []
md_content.append("# Data Profiling and Preliminary Analysis Report\n\n")

## Automatic Generation of the Schema

In [ ]:
md_content.append("## 1. Data Dictionary\n\n")

for file_name in ORIGINAL_FILES:
    file_path = os.path.join(ORIGINAL_DATA_DIR, file_name)
    if os.path.exists(file_path):
        temp_df = pd.read_csv(file_path, nrows=1)
        temp_df.columns = [c.strip().replace(" ", "_").upper() for c in temp_df.columns]

        md_content.append(f"### {file_name.upper()}\n\n")
        md_content.append("| Column | Type | Description |\n")
        md_content.append("| :--- | :--- | :--- |\n")

        for col in temp_df.columns:
            dtype = str(temp_df[col].dtype)
            desc = COLUMN_DESCRIPTIONS.get(col, "N/A")
            md_content.append(f"| {col} | {dtype} | {desc} |\n")
        md_content.append("\n")

print("Tables generated for Data Dictionary.")

## Load Main Dataset

In [ ]:
# Load all original CSVs at once using the shared loader
# normalize_cols='upper' preserves the UPPERCASE convention of the raw dataset
dfs = load_tables("original", normalize_cols="upper")

# Register schemas for post-operation validation
contract = SchemaContract()
for name, df in dfs.items():
    contract.register(name, df)

# Quick schema integrity report
print("\nRegistered schemas:")
for t in contract.registered_tables:
    print(f"  {t}")

# Main dataset is CompleteData
df = dfs.get("CompleteData")
if df is not None:
    print(f"\nMain dataset: {len(df):,} rows x {len(df.columns)} cols")

    md_content.append("## 2. Main Dataset Overview\n")
    md_content.append("- Analyzed file: CompleteData.csv\n")
    md_content.append(f"- Total rows: {len(df):,}\n")
    md_content.append(f"- Total columns: {len(df.columns)}\n\n")
else:
    raise FileNotFoundError("CompleteData.csv not found - check ORIGINAL_DATA_DIR in config")

## Functional Dependencies (Weather)

In [ ]:
group_cols = ["MESONET_STATION", "FL_DATE", "DEP_HOUR"]

print("Checking dependency: (MESONET_STATION, FL_DATE, DEP_HOUR) -> weather attributes")
grouped = df.groupby(group_cols)[ORIGINAL_WEATHER_COLS].nunique(dropna=False)
violations = grouped[(grouped > 1).any(axis=1)]

md_content.append("## 3. Functional Dependencies Analysis\n")
md_content.append("### (MESONET_STATION, FL_DATE, DEP_HOUR) -> Weather Attributes\n")
md_content.append(f"- Total groups analyzed: {len(grouped):,}\n")
md_content.append(f"- Violations found: {len(violations):,}\n")

if len(violations) == 0:
    md_content.append("- Result: SUCCESS. The functional dependency is valid across the dataset.\n\n")
    print("-> SUCCESS")
else:
    md_content.append("- Result: WARNING. There are inconsistencies in weather values within identical stations and hours.\n\n")
    print("-> WARNING")

## Station Mapping (Origin vs Dest)

In [ ]:
print("Checking mapping between Airports and Weather Stations...")

# 1. ORIGIN -> MESONET_STATION
origin_to_station = df.groupby("ORIGIN")["MESONET_STATION"].nunique(dropna=False)
viol_origin = origin_to_station[origin_to_station > 1]

# 2. MESONET_STATION -> ORIGIN
station_to_origin = df.groupby("MESONET_STATION")["ORIGIN"].nunique(dropna=False)
viol_station = station_to_origin[station_to_origin > 1]

# 3. DEST -> MESONET_STATION (Exclusion Check)
dest_to_station = df.groupby("DEST")["MESONET_STATION"].nunique(dropna=False)
viol_dest = dest_to_station[dest_to_station > 1]

md_content.append("### Geographic Mapping: Airports and Weather Stations\n")
md_content.append(f"- ORIGIN to MESONET_STATION: Origin airports mapped to multiple stations: {len(viol_origin):,} out of {len(origin_to_station):,}\n")
md_content.append(f"- MESONET_STATION to ORIGIN: Stations mapped to multiple origin airports: {len(viol_station):,} out of {len(station_to_origin):,}\n")

md_content.append("\nExclusion Test (DEST vs MESONET_STATION)\n")
md_content.append(f"- DEST to MESONET_STATION: Destinations mapped to multiple stations: {len(viol_dest):,} out of {len(dest_to_station):,}. \n")
md_content.append(
    "Note: The very high percentage of violations on DEST, opposed to the uniqueness on ORIGIN, provides empirical certainty that the MESONET_STATION attribute recorded in the flight record refers exclusively to the weather conditions of the departure airport.\n\n"
)

print("Mapping calculated and recorded.")

## Cancelled Flights Consitency Analysis

In [ ]:
print("Checking cancelled flights consistency...")
cancelled_df = df[df["CANCELLED"] != 0].copy()

md_content.append("## 4. Consistency Analysis (Cancelled Flights)\n")
md_content.append(f"- Total cancelled flights analyzed: {len(cancelled_df):,}\n")

if len(cancelled_df) > 0:
    cancelled_df["FL_DATE"] = pd.to_datetime(cancelled_df["FL_DATE"])
    cancelled_df["DEP_TIME"] = pd.to_datetime(cancelled_df["DEP_TIME"], errors="coerce")

    expected_dep_time = cancelled_df["FL_DATE"] + pd.to_timedelta(cancelled_df["DEP_HOUR"], unit="h")
    midnight_mask = (cancelled_df["DEP_HOUR"] == 0) & (cancelled_df["DEP_TIME"].dt.hour == 0) & (cancelled_df["DEP_TIME"].dt.date == (cancelled_df["FL_DATE"] + pd.Timedelta(days=1)).dt.date)
    expected_dep_time.loc[midnight_mask] += pd.Timedelta(days=1)

    dep_time_null = cancelled_df["DEP_TIME"].isna().sum()
    non_null_dep = cancelled_df["DEP_TIME"].notna()
    dep_matches = cancelled_df.loc[non_null_dep, "DEP_TIME"].dt.floor("h") == expected_dep_time[non_null_dep].dt.floor("h")
    matching_dep = dep_matches.sum()
    total_non_null_dep = non_null_dep.sum()

    dep_delay_null = cancelled_df["DEP_DELAY"].isna().sum()
    dep_delay_zero = (cancelled_df["DEP_DELAY"] == 0).sum()

    operated_cancelled = cancelled_df[((cancelled_df["AIR_TIME"].fillna(0) > 0) | (cancelled_df["TAXI_OUT"].fillna(0) > 0))]
    cancelled_with_airtime = cancelled_df[cancelled_df["AIR_TIME"].fillna(0) > 0]

    md_content.append("\n### Departure Metrics (DEP_TIME and DEP_DELAY)\n")
    md_content.append(f"- DEP_TIME NULL: {dep_time_null:,} ({100 * dep_time_null / len(cancelled_df):.2f}%)\n")
    if total_non_null_dep > 0:
        md_content.append(f"- DEP_TIME aligned to FL_DATE and DEP_HOUR: {matching_dep:,} / {total_non_null_dep:,} ({100 * matching_dep / total_non_null_dep:.2f}%)\n")

    md_content.append(f"- DEP_DELAY NULL: {dep_delay_null:,} ({100 * dep_delay_null / len(cancelled_df):.2f}%)\n")
    md_content.append(f"- DEP_DELAY equals 0: {dep_delay_zero:,} ({100 * dep_delay_zero / len(cancelled_df):.2f}%)\n")

    md_content.append("\n### Operational Anomalies Checks\n")
    md_content.append(f"- Cancelled flights with AIR_TIME > 0 or TAXI_OUT > 0: {len(operated_cancelled):,}\n")
    md_content.append(f"- Cancelled flights with only AIR_TIME > 0 recorded: {len(cancelled_with_airtime):,}\n")
    if len(operated_cancelled) == 0:
        md_content.append("Result: No anomalies. Cancelled flights do not show operational taxi or flight times.\n\n")

print("Analysis completed.")

## Post-Analysis Schema Validation

In [ ]:
# Verify that no structural changes occurred during analysis
report = contract.validate_all(dfs)
if report["failures"] == 0:
    print("Schema validation PASSED - all tables unchanged after analysis.")
else:
    print("Schema validation FAILED - unexpected structural changes detected!")
    for entry in report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")

## Write Output File

In [ ]:
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.writelines(md_content)

print(f"--- ANALYSIS COMPLETED: Report saved successfully as '{REPORT_FILE}' ---")